# 01 · Built environment vs. car ownership

**Question this notebook answers (RQ part 1):** How strongly is mixed-use development associated with lower car
dependence across U.S. neighborhoods, and how does that compare with population density, street connectivity,
and transit access?

**Data:** EPA Smart Location Database v3.0, one row per Census block group, cleaned by `src/build_sld.py`.

**Sample:** block groups in metro areas of 1M+ people (our "major cities") with at least 50 households.

**Car dependence measure:** share of households owning two or more cars (`pct_2plus_cars`). Averages are
*household-weighted*, so each point reads as "the share of households in these neighborhoods that own 2+ cars."

In [ ]:
import numpy as np
import pandas as pd
import altair as alt
from pathlib import Path

Path("../figures").mkdir(exist_ok=True)

sld = pd.read_csv("../data/processed/sld_blockgroups.csv.gz", dtype={"geoid": str, "cbsa": str})
df = sld[sld["major_metro"] & sld["enough_households"]].copy()

print(f"{len(df):,} block groups in {df['cbsa'].nunique()} major metros")
df[["pct_2plus_cars", "mixed_use", "pop_density", "intersection_density", "transit_dist_m"]].describe().round(2)

## Step 1 · Put the four features on a common scale

The four features are in different units (an entropy score, people/acre, intersections/sq mi, meters), so we
can't compare them directly. Instead we rank block groups on each feature and convert to a **percentile**
(0 = most car-oriented, 100 = most urban). Then we group into ten bins and compute the 2+ car share in each bin.
If one feature's line drops more steeply, that feature separates high-car and low-car neighborhoods more strongly.

*Transit:* the SLD records no distance when the nearest stop is more than ¾ mile away. Those block groups are all
tied at "worst access", so they form one point (labeled on the chart) instead of being spread over several bins.

In [ ]:
# Transit access: closer = better, and "no stop within 3/4 mile" = worst possible.
df["transit_access"] = -df["transit_dist_m"]
df.loc[df["no_stop_within_3_4mi"], "transit_access"] = -np.inf

FEATURES = {                      # column -> label shown on charts
    "mixed_use": "Mixed use",
    "pop_density": "Population density",
    "intersection_density": "Street connectivity",
    "transit_access": "Transit access",
}
OUTCOME = "pct_2plus_cars"


def profile(data, col, label, outcome=OUTCOME, n_bins=10):
    """Household-weighted outcome by percentile bin of `col`."""
    d = data[[col, outcome, "households"]].dropna().copy()
    d["pct"] = d[col].rank(pct=True, method="average") * 100
    d["bin"] = np.ceil(d["pct"] / (100 / n_bins)).clip(1, n_bins).astype(int)
    d["w"] = d[outcome] * d["households"]
    g = d.groupby("bin").agg(x=("pct", "mean"), w=("w", "sum"), hh=("households", "sum"), n=(outcome, "size"))
    g["y"] = g["w"] / g["hh"]
    return g.reset_index().assign(feature=label)[["feature", "bin", "x", "y", "n"]]


prof = pd.concat([profile(df, c, lab) for c, lab in FEATURES.items()], ignore_index=True)
prof.round(1)

## Step 2 · Figure 1: which feature tracks car ownership most closely?

In [ ]:
PALETTE = {
    "Mixed use": "#d1495b",            # our focus variable: strongest color
    "Population density": "#3a6ea5",
    "Street connectivity": "#2a9d8f",
    "Transit access": "#8d6cab",
}
color = alt.Color("feature:N", scale=alt.Scale(domain=list(PALETTE), range=list(PALETTE.values())), legend=None)

base = alt.Chart(prof).encode(
    x=alt.X("x:Q", scale=alt.Scale(domain=[0, 100]),
            title="Percentile of block groups on each feature (0 = most car-oriented, 100 = most urban)"),
    y=alt.Y("y:Q", scale=alt.Scale(zero=False), title="Households owning 2+ cars (%)"),
    color=color,
)
lines = base.mark_line(strokeWidth=2).transform_filter(alt.datum.feature != "Mixed use")
focus = base.mark_line(strokeWidth=3.5).transform_filter(alt.datum.feature == "Mixed use")
points = base.mark_circle(size=45, opacity=1).encode(
    tooltip=["feature", alt.Tooltip("y:Q", format=".1f", title="2+ cars (%)"), alt.Tooltip("n:Q", format=",", title="block groups")]
)
# End-of-line labels, nudged apart so they don't overlap when two lines finish close together.
ends = prof[prof["bin"] == 10].sort_values("y").reset_index(drop=True)
min_gap = (prof["y"].max() - prof["y"].min()) * 0.06
ends["label_y"] = ends["y"]
for i in range(1, len(ends)):
    ends.loc[i, "label_y"] = max(ends.loc[i, "y"], ends.loc[i - 1, "label_y"] + min_gap)
end_labels = alt.Chart(ends).mark_text(align="left", dx=10, fontSize=12, fontWeight="bold").encode(
    x="x:Q", y="label_y:Q", text="feature:N", color=color)

# Label the single "no stop within 3/4 mile" transit point.
no_stop = prof[(prof["feature"] == "Transit access")].nsmallest(1, "x").assign(label="no stop within ¾ mi")
no_stop_label = alt.Chart(no_stop).mark_text(align="left", dx=-6, dy=16, fontSize=11, color="#8d6cab").encode(
    x="x:Q", y="y:Q", text="label:N")

fig1 = (lines + focus + points + end_labels + no_stop_label).properties(
    width=520, height=330,
    title=alt.TitleParams("Which neighborhood features track car ownership most closely?",
                          subtitle=f"Block groups in U.S. metros of 1M+ people (n = {len(df):,}); household-weighted",
                          anchor="start"),
).configure_view(stroke=None).configure_axis(grid=True, gridOpacity=0.4)

fig1.save("../figures/fig1_feature_comparison.png", scale_factor=2)
fig1.save("../figures/fig1_feature_comparison.html")
fig1

**Numbers for the write-up.** `gap` is how many percentage points the 2+ car share falls from the
least to the most urban bin of each feature. `spearman` is the rank correlation with the 2+ car share
(more negative = stronger association).

In [ ]:
summary = (prof.sort_values("x").groupby("feature")
           .agg(least_urban=("y", "first"), most_urban=("y", "last")))
summary["gap"] = summary["least_urban"] - summary["most_urban"]
column_of = {label: col for col, label in FEATURES.items()}
summary["spearman"] = [df[column_of[f]].corr(df[OUTCOME], method="spearman") for f in summary.index]
summary.sort_values("gap", ascending=False).round(2)

## Step 3 · Figure 2: does mixed use still matter at similar densities?

Dense places also tend to be mixed, connected, and transit-served, so Figure 1 can't tell us whether mixed use
matters *on its own*. Here we split block groups into density terciles and redo the comparison **within** each
tercile (percentiles are computed inside each tercile). If a feature's lines stay steep, it is associated with
car ownership even among neighborhoods of similar density; if they go flat, its Figure 1 slope was mostly density.

In [ ]:
df["density_group"] = pd.qcut(df["pop_density"], 3, labels=["Low density", "Medium density", "High density"])

prof2 = pd.concat([
    profile(grp, c, lab).assign(density_group=dg)
    for dg, grp in df.groupby("density_group", observed=True)
    for c, lab in FEATURES.items() if c != "pop_density"
], ignore_index=True)

fig2 = alt.Chart(prof2).mark_line(point=True, strokeWidth=2.5).encode(
    x=alt.X("x:Q", scale=alt.Scale(domain=[0, 100]), title="Percentile within density group"),
    y=alt.Y("y:Q", scale=alt.Scale(zero=False), title="Households owning 2+ cars (%)"),
    color=alt.Color("density_group:N", sort=["Low density", "Medium density", "High density"],
                    scale=alt.Scale(range=["#9ecae1", "#4292c6", "#08306b"]),
                    legend=alt.Legend(title=None, orient="bottom")),
    tooltip=["feature", "density_group", alt.Tooltip("y:Q", format=".1f"), alt.Tooltip("n:Q", format=",")],
).properties(width=210, height=240).facet(
    column=alt.Column("feature:N", sort=["Mixed use", "Street connectivity", "Transit access"], title=None,
                      header=alt.Header(labelFontSize=13, labelFontWeight="bold")),
).properties(
    title=alt.TitleParams("Holding density roughly fixed, which features still track car ownership?",
                          subtitle="Lines = density terciles. x-axis: 0 = most car-oriented, 100 = most urban within that tercile. Steeper = stronger.",
                          anchor="start"),
).configure_view(stroke=None)

fig2.save("../figures/fig2_within_density.png", scale_factor=2)
fig2.save("../figures/fig2_within_density.html")
fig2

In [ ]:
gaps = (prof2.sort_values("x").groupby(["feature", "density_group"], observed=True)["y"]
        .agg(lambda s: s.iloc[0] - s.iloc[-1]).unstack().round(1)
        [["Low density", "Medium density", "High density"]])
gaps.columns.name = "gap (pct pts) by density group"
gaps

## Next steps (not needed for the proposal)

* **Income:** repeat Figure 2 splitting by income (ACS median household income, once joined on `geoid`).
* **Commuting:** swap the outcome for ACS drive-alone / transit / walk shares (table B08301).
* **CO₂ (RQ part 2):** join Vulcan on-road emissions to block groups, then plot emissions per resident and per km²
  against the same feature percentiles.